### Introducción a Unity Catalog

Unity Catalog es un catálogo de datos centralizado que proporciona funcionalidades de control de acceso, auditoría, linaje, supervisión de calidad y detección de datos en áreas de trabajo de Databricks.

Las principales características de Unity Catalog incluyen:

* **Definir una vez, proteger en todas partes:** El catálogo de Unity ofrece un único lugar para administrar directivas de acceso a datos que se aplican en todas las áreas de trabajo de una región.
* **Modelo de seguridad compatible con estándares:** el modelo de seguridad de Unity Catalog se basa en ANSI SQL estándar y permite a los administradores conceder permisos en su lago de datos existente mediante una sintaxis conocida.
* **Auditoría integrada y linaje:** Unity Catalog captura automáticamente los registros de auditoría de nivel de usuario que registran el acceso a los datos. El Catálogo Unity también captura datos de linaje que rastrean cómo se crean y utilizan los activos de datos en todos los idiomas.
* **Detección de datos:** Unity Catalog permite etiquetar y documentar recursos de datos y proporciona una interfaz de búsqueda para ayudar a los consumidores de datos a encontrar datos.
* **Tablas del sistema:** El catálogo de Unity le permite acceder y consultar fácilmente los datos operativos de su cuenta, incluidos los registros de auditoría, el uso facturable y el linaje.

In [0]:
from pyspark.sql import functions as F

#### El modelo de objetos de Catálogo de Unity
En un metastore de Unity Catalog, la jerarquía de objetos de base de datos de tres niveles consta de catálogos que contienen esquemas, que a su vez contienen datos y objetos de inteligencia artificial, como tablas y modelos. Esta jerarquía se representa como un espacio de nombres de tres niveles (catalog.schema.table-etc) al hacer referencia a tablas, vistas, volúmenes, modelos y funciones.

![imagen_1771904183725.png](./imagen_1771904183725.png "imagen_1771904183725.png")

#### Catálogos
Los catálogos se usan para organizar los recursos de datos y normalmente se usan como nivel superior en el esquema de aislamiento de datos. Los catálogos suelen reflejar las unidades organizativas o los ámbitos del ciclo de vida de desarrollo de software.

In [0]:
%sql
create catalog if not exists dev;

#### Esquemas 
Los **esquemas** (también conocidos como bases de datos) contienen tablas, vistas, volúmenes, modelos de IA y funciones. Los esquemas organizan los datos y los recursos de IA en categorías lógicas que son más granulares que los catálogos. Normalmente, un esquema representa un único caso de uso, proyecto o espacio aislado de equipo.

In [0]:
%sql
create database if not exists dev.ciencias_data

#### Volúmenes
Los volúmenes representan volúmenes lógicos de datos en el almacenamiento de objetos en la nube. Puede usar volúmenes para almacenar y organizar archivos en cualquier formato, incluidos datos estructurados, semiestructurados y no estructurados, así como acceder a ellos. Normalmente se usan para datos no tabulares. Los volúmenes pueden ser administrados, con Unity Catalog gestionando el ciclo de vida completo y el diseño de los datos en el almacenamiento, o externos, con Unity Catalog gestionando el acceso a los datos desde Databricks, pero sin gestionar el acceso a los datos en la nube desde otros clientes.

In [0]:
%sql
create volume if not exists dev.ciencias_data.session_data;

In [0]:
df=spark.read.format("csv").option("sep","|").option("header","true").load("/Volumes/dev/ciencias_data/session_data/sessions_part2.csv")
df=df.withColumn("_load_timestamp",F.lit(F.current_timestamp())).withColumn("_source",F.lit("Arkime"))
df.display()

#### Tablas 
Las tablas son colecciones de datos organizados por filas y columnas. Las tablas se pueden administrar, con Unity Catalog gestionando el ciclo de vida completo de la tabla o ser externas, con Unity Catalog administrando el acceso a los datos desde Databricks, pero sin gestionar el acceso a los datos del almacenamiento en la nube desde otros clientes.

In [0]:
df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("dev.ciencias_data.sesions_part1")

In [0]:
df.write.format("delta").save("/Volumes/dev/ciencias_data/session_data/delta_table/sesiones")

In [0]:
df_log=spark.read.format("json").load("/Volumes/dev/ciencias_data/session_data/delta_table/sesiones/_delta_log/00000000000000000000.json")
df_log.display()

In [0]:
[x.name for x in dbutils.fs.ls("dbfs:/Volumes/dev/ciencias_data/session_data/delta_table/sesiones")]

In [0]:
%sql
select * from dev.ciencias_data.sesions_part1 limit 10

In [0]:
from pyspark.sql.types import StructType,StringType,StructField,IntegerType,ArrayType,LongType
schema=StructType([
    StructField("srcIp",StringType()),
    StructField("dstIp",StringType()),
    StructField("srcMac",StringType()),
    StructField("dstDataBytes",IntegerType()),
    StructField("lastPacket",LongType()),
    StructField("firstPacket",LongType()),
    StructField("dstBytes",IntegerType()),
    StructField("packetLen",ArrayType(IntegerType()))
    ])
df_slv=spark.sql("select * from dev.ciencias_data.sesions_part1")
df_slv=df_slv.withColumn("data_struct",F.from_json(F.col("data"),schema))
df_slv=df_slv.select(F.col("data_struct.dstDataBytes").alias("dst_data_bytes"),F.col("data_struct.dstBytes").alias("dst_bytes"),F.col("data_struct.srcIp"),F.col("data_struct.dstIp"),F.col("data_struct.srcMac"),F.col("data_struct.lastPacket").alias("last_packet"),F.col("data_struct.firstPacket").alias("first_packet"))
df_final=df_slv.withColumn("session_duration",F.col("last_packet")-F.col("first_packet"))
df_final.display()

In [0]:
df_final.write.format("delta").saveAsTable("dev.ciencias_data.slv_sessions")